# Time Travel & Data Recovery (Day 11)

## Objective
Learn how to use **Delta Lake Time Travel** for:
* **Versioning**: Track every change to your data
* **Rollback**: Restore previous versions
* **Data Recovery**: Recover accidentally deleted/modified data
* **Auditing**: Review historical data states

## What is Delta Lake Time Travel?

Every write operation to a Delta table creates a new **version**:
* **Version 0**: Initial table creation
* **Version 1**: First INSERT/UPDATE/DELETE
* **Version 2**: Second operation
* ...

You can query **any historical version** using:
```python
# Query by version number
spark.read.format("delta").option("versionAsOf", 0).load("/path/to/table")

# Query by timestamp
spark.read.format("delta").option("timestampAsOf", "2021-01-01").load("/path/to/table")
```

## Use Cases
✅ **Auditing**: "What did the data look like yesterday?"  
✅ **Rollback**: "Undo the last update that broke the pipeline"  
✅ **A/B Testing**: "Compare model performance on version 5 vs version 10"  
✅ **Compliance**: "Reproduce reports from 6 months ago"  
✅ **Debugging**: "When did this record change?"

## Challenge Tasks (Day 11)
✅ Append new records  
✅ Query older version  
✅ Compare differences

In [0]:
# Configuration
CATALOG = "dbacademy"
SCHEMA = "labuser13955035_1772876383"
BASE_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_model_scoring_base"

print(f"📅 Time Travel Target: {BASE_TABLE}")
print(f"🎯 Objective: Explore Delta Lake versioning")
print("="*80)

In [0]:
# View version history of Delta table
print("📋 DELTA TABLE HISTORY")
print("="*80)

history_df = spark.sql(f"DESCRIBE HISTORY {BASE_TABLE}")

print(f"\nTotal versions: {history_df.count()}")
print("\nRecent versions:")
display(history_df.select(
    "version",
    "timestamp",
    "operation",
    "operationMetrics.numOutputRows",
    "operationMetrics.numFiles"
).limit(10))

print("""
💡 Key Columns:
• version: Sequential version number
• timestamp: When this version was created
• operation: Type of operation (CREATE, WRITE, UPDATE, DELETE, OPTIMIZE)
• operationMetrics: Rows added/removed, files changed
• userMetadata: Custom tags/notes
""")

## Delta Table Versions

Each operation creates a new version:

| Version | Operation | Description |
| --- | --- | --- |
| 0 | CREATE TABLE | Initial table creation |
| 1 | INSERT | First data load |
| 2 | UPDATE | Modified existing rows |
| 3 | DELETE | Removed rows |
| 4 | MERGE | Upsert operation |
| 5 | OPTIMIZE | File compaction |
| 6 | RESTORE | Rollback to version 3 |

## Retention Period

By default, Delta Lake retains version history for:
* **30 days** (configurable via `delta.logRetentionDuration`)
* After 30 days, old versions are **vacuumed** (deleted)

### Configure Retention
```sql
ALTER TABLE table_name SET TBLPROPERTIES (
  'delta.logRetentionDuration' = '90 days',
  'delta.deletedFileRetentionDuration' = '90 days'
)
```

In [0]:
# Query current version (default behavior)
print("🔴 CURRENT VERSION (Latest)")
print("="*80)

current_df = spark.table(BASE_TABLE)

print(f"Total records: {current_df.count():,}")
print(f"Date range: {current_df.selectExpr('MIN(event_time_utc)', 'MAX(event_time_utc)').collect()[0]}")
print(f"\nPrice statistics:")
display(current_df.selectExpr(
    "COUNT(*) as total_records",
    "ROUND(AVG(price_eur_mwh), 2) as avg_price",
    "ROUND(MIN(price_eur_mwh), 2) as min_price",
    "ROUND(MAX(price_eur_mwh), 2) as max_price"
))

In [0]:
# Query a specific historical version
print("🔵 HISTORICAL VERSION (Version 0 - Original Data)")
print("="*80)

# Get version 0 (original table creation)
try:
    historical_df = spark.read \
        .format("delta") \
        .option("versionAsOf", 0) \
        .load(f"/user/hive/warehouse/{CATALOG}.db/{SCHEMA}.db/{BASE_TABLE.split('.')[-1]}")
    
    print(f"Total records in version 0: {historical_df.count():,}")
    print(f"Date range: {historical_df.selectExpr('MIN(event_time_utc)', 'MAX(event_time_utc)').collect()[0]}")
    print(f"\nPrice statistics (version 0):")
    display(historical_df.selectExpr(
        "COUNT(*) as total_records",
        "ROUND(AVG(price_eur_mwh), 2) as avg_price",
        "ROUND(MIN(price_eur_mwh), 2) as min_price",
        "ROUND(MAX(price_eur_mwh), 2) as max_price"
    ))
except Exception as e:
    print(f"⚠️ Could not query version 0: {e}")
    print("💡 This is normal if the table was created with all data at once (no incremental versions)")
    
    # Alternative: Query by timestamp
    print("\n🔵 ALTERNATIVE: Query by timestamp (1 week ago)")
    from datetime import datetime, timedelta
    one_week_ago = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")
    print(f"Querying data as of: {one_week_ago}")
    
    try:
        timestamp_df = spark.read \
            .format("delta") \
            .option("timestampAsOf", one_week_ago) \
            .table(BASE_TABLE)
        print(f"Records as of {one_week_ago}: {timestamp_df.count():,}")
    except Exception as e2:
        print(f"⚠️ Could not query by timestamp: {e2}")
        print("💡 Table may not have versions from that date")

In [0]:
# Create a test table to demonstrate time travel
print("🧪 CREATING TEST TABLE FOR TIME TRAVEL DEMO")
print("="*80)

TEST_TABLE = f"{CATALOG}.{SCHEMA}.h2_time_travel_demo"

# Step 1: Create initial version (Version 0)
print("\n➡️ Version 0: Creating table with 2020 data only")
initial_data = spark.table(BASE_TABLE).filter("YEAR(event_time_utc) = 2020")
initial_data.write.format("delta").mode("overwrite").saveAsTable(TEST_TABLE)

v0_count = spark.table(TEST_TABLE).count()
print(f"✅ Version 0 created: {v0_count:,} records (2020 only)")

# Step 2: Append 2021 data (Version 1)
print("\n➡️ Version 1: Appending 2021 data")
append_data = spark.table(BASE_TABLE).filter("YEAR(event_time_utc) = 2021")
append_data.write.format("delta").mode("append").saveAsTable(TEST_TABLE)

v1_count = spark.table(TEST_TABLE).count()
print(f"✅ Version 1 created: {v1_count:,} records (2020 + 2021)")

# Step 3: Update prices (Version 2)
print("\n➡️ Version 2: Updating prices (apply 10% markup)")
spark.sql(f"""
    UPDATE {TEST_TABLE}
    SET price_eur_mwh = price_eur_mwh * 1.10
    WHERE YEAR(event_time_utc) = 2021
""")

v2_avg_price = spark.table(TEST_TABLE).filter("YEAR(event_time_utc) = 2021").selectExpr("AVG(price_eur_mwh)").collect()[0][0]
print(f"✅ Version 2 created: Prices updated (2021 avg: {v2_avg_price:.2f} EUR/MWh)")

print("\n✅ Test table created with 3 versions!")
print(f"Table: {TEST_TABLE}")

In [0]:
# Compare different versions of the test table
print("🔍 COMPARING VERSIONS")
print("="*80)

# Version 0: Original (2020 only)
print("\n🔵 VERSION 0: Original data")
v0_df = spark.read.format("delta").option("versionAsOf", 0).table(TEST_TABLE)
v0_stats = v0_df.selectExpr(
    "COUNT(*) as records",
    "MIN(YEAR(event_time_utc)) as min_year",
    "MAX(YEAR(event_time_utc)) as max_year",
    "ROUND(AVG(price_eur_mwh), 2) as avg_price"
).collect()[0]
print(f"Records: {v0_stats[0]:,}")
print(f"Years: {v0_stats[1]} - {v0_stats[2]}")
print(f"Avg Price: {v0_stats[3]} EUR/MWh")

# Version 1: After append (2020 + 2021)
print("\n🟡 VERSION 1: After appending 2021 data")
v1_df = spark.read.format("delta").option("versionAsOf", 1).table(TEST_TABLE)
v1_stats = v1_df.selectExpr(
    "COUNT(*) as records",
    "MIN(YEAR(event_time_utc)) as min_year",
    "MAX(YEAR(event_time_utc)) as max_year",
    "ROUND(AVG(price_eur_mwh), 2) as avg_price"
).collect()[0]
print(f"Records: {v1_stats[0]:,}")
print(f"Years: {v1_stats[1]} - {v1_stats[2]}")
print(f"Avg Price: {v1_stats[3]} EUR/MWh")

# Version 2: After update (prices increased)
print("\n🔴 VERSION 2: After price update (+10% for 2021)")
v2_df = spark.table(TEST_TABLE)  # Current version
v2_stats = v2_df.selectExpr(
    "COUNT(*) as records",
    "MIN(YEAR(event_time_utc)) as min_year",
    "MAX(YEAR(event_time_utc)) as max_year",
    "ROUND(AVG(price_eur_mwh), 2) as avg_price"
).collect()[0]
print(f"Records: {v2_stats[0]:,}")
print(f"Years: {v2_stats[1]} - {v2_stats[2]}")
print(f"Avg Price: {v2_stats[3]} EUR/MWh (+10% markup applied)")

# Summary
print("\n" + "="*80)
print("📊 SUMMARY")
print(f"Version 0 → 1: Added {v1_stats[0] - v0_stats[0]:,} records")
print(f"Version 1 → 2: Price increased by {((v2_stats[3] - v1_stats[3]) / v1_stats[3] * 100):.1f}%")

## Time Travel Syntax Options

### 1. Query by Version Number
```python
# Python API
df = spark.read.format("delta").option("versionAsOf", 5).table("table_name")

# SQL
SELECT * FROM table_name VERSION AS OF 5
```

### 2. Query by Timestamp
```python
# Python API
df = spark.read.format("delta").option("timestampAsOf", "2021-12-31").table("table_name")

# SQL
SELECT * FROM table_name TIMESTAMP AS OF '2021-12-31 23:59:59'
```

### 3. Query by Date
```python
# SQL with date
SELECT * FROM table_name TIMESTAMP AS OF '2021-12-31'
```

### 4. Using @ Syntax (SQL)
```sql
SELECT * FROM table_name@v5              -- Version 5
SELECT * FROM table_name@20211231        -- Date
```

## Best Practices

✅ **Use version numbers** for exact reproducibility  
✅ **Use timestamps** for "data as of X date" queries  
✅ **Document versions** in experiment metadata  
❌ **Don't rely on time travel forever** (vacuumed after 30 days by default)

In [0]:
# Simulate data recovery scenario
print("🚑 DATA RECOVERY SCENARIO")
print("="*80)

# Scenario: Accidentally deleted high-price records
print("\n⚠️ ACCIDENT: Deleting high-price records (> 150 EUR/MWh)")
deleted_count_before = spark.table(TEST_TABLE).filter("price_eur_mwh > 150").count()
print(f"Records to be deleted: {deleted_count_before:,}")

spark.sql(f"""
    DELETE FROM {TEST_TABLE}
    WHERE price_eur_mwh > 150
""")

deleted_count_after = spark.table(TEST_TABLE).filter("price_eur_mwh > 150").count()
print(f"Records after DELETE: {deleted_count_after:,}")
print(f"🔴 DELETED: {deleted_count_before - deleted_count_after:,} records!")

# Recovery: Query previous version
print("\n🔵 RECOVERY: Querying version before deletion (Version 2)")
recovered_df = spark.read.format("delta").option("versionAsOf", 2).table(TEST_TABLE)
recovered_count = recovered_df.filter("price_eur_mwh > 150").count()
print(f"Records in version 2 (before delete): {recovered_count:,}")

# Option 1: Restore using RESTORE TABLE command
print("\n✅ OPTION 1: Restore entire table to version 2")
print(f"SQL: RESTORE TABLE {TEST_TABLE} TO VERSION AS OF 2")

# Option 2: Re-insert deleted records
print("\n✅ OPTION 2: Re-insert deleted records only")
print("Python:")
print(f"""
deleted_records = spark.read.format("delta").option("versionAsOf", 2).table("{TEST_TABLE}") \\
    .filter("price_eur_mwh > 150")
deleted_records.write.format("delta").mode("append").saveAsTable("{TEST_TABLE}")
""")

print("\n💡 Time Travel enables easy data recovery!")

In [0]:
# Restore table to previous version
print("⏪ ROLLBACK: Restore table to version 2")
print("="*80)

print(f"\nCurrent version: {spark.sql(f'DESCRIBE HISTORY {TEST_TABLE}').select('version').first()[0]}")
print(f"Current record count: {spark.table(TEST_TABLE).count():,}")

print("\n➡️ Restoring to version 2...")
spark.sql(f"RESTORE TABLE {TEST_TABLE} TO VERSION AS OF 2")

print(f"\n✅ Restored!")
print(f"Record count after restore: {spark.table(TEST_TABLE).count():,}")
print(f"New version: {spark.sql(f'DESCRIBE HISTORY {TEST_TABLE}').select('version').first()[0]}")

print("""

💡 KEY INSIGHT:
RESTORE creates a NEW version that is a copy of the old version.
It does NOT delete versions - you can still access version 3 (the delete operation).
""")

In [0]:
# Analyze change history for auditing
print("🔍 AUDIT TRAIL ANALYSIS")
print("="*80)

print(f"\nFull version history of {TEST_TABLE}:")
history = spark.sql(f"DESCRIBE HISTORY {TEST_TABLE}")
display(history.select(
    "version",
    "timestamp",
    "operation",
    "operationParameters",
    "operationMetrics"
))

print("""

📊 AUDIT INSIGHTS:

1. ACCOUNTABILITY
   • Every change is logged with timestamp
   • Operation type tracked (INSERT, UPDATE, DELETE, RESTORE)
   • User metadata can be added for compliance

2. REPRODUCIBILITY
   • Reproduce any historical state
   • Compare model performance across data versions
   • Debug pipeline issues by checking historical data

3. COMPLIANCE
   • Regulatory requirements (e.g., GDPR right to explanation)
   • Financial audits ("show me Q2 2021 data")
   • Data lineage tracking
""")

## Advanced Time Travel Use Cases

### 1. A/B Testing with Data Versions
```python
# Train model A on version 5
data_v5 = spark.read.format("delta").option("versionAsOf", 5).table("training_data")
model_a = train_model(data_v5)

# Train model B on version 10 (after feature engineering)
data_v10 = spark.read.format("delta").option("versionAsOf", 10).table("training_data")
model_b = train_model(data_v10)

# Compare performance
```

### 2. Schema Evolution Tracking
```python
# Check schema changes over time
for version in [0, 5, 10]:
    df = spark.read.format("delta").option("versionAsOf", version).table("table")
    print(f"Version {version} schema: {df.schema}")
```

### 3. Data Quality Monitoring
```python
# Compare data quality metrics across versions
from pyspark.sql.functions import count, avg, stddev

for version in range(0, 5):
    df = spark.read.format("delta").option("versionAsOf", version).table("table")
    stats = df.agg(
        count("*").alias("total"),
        avg("price").alias("avg_price"),
        stddev("price").alias("price_stddev")
    ).collect()[0]
    print(f"Version {version}: {stats}")
```

### 4. Incremental ETL Verification
```python
# Verify incremental loads
v_before = spark.read.format("delta").option("versionAsOf", 5).table("table").count()
v_after = spark.read.format("delta").option("versionAsOf", 6).table("table").count()
print(f"Records added in version 6: {v_after - v_before}")
```

In [0]:
# Clean up test table
print("🧹 CLEANUP")
print("="*80)

print(f"\nDropping test table: {TEST_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {TEST_TABLE}")
print("✅ Test table dropped")

print("""

💡 IMPORTANT: In production, use VACUUM to clean up old files:

VACUUM table_name RETAIN 168 HOURS  -- Keep 7 days
VACUUM table_name DRY RUN            -- Preview files to delete

VACUUM permanently deletes old data files.
After VACUUM, you can only time travel to versions within the retention period.
""")

In [0]:
print("📊 TIME TRAVEL & DATA RECOVERY SUMMARY (Day 11)")
print("="*80)

print("""
✅ COMPLETED CHALLENGE TASKS:
1. ✅ Append new records (Version 1: added 2021 data)
2. ✅ Query older version (Version 0: original data)
3. ✅ Compare differences (Versions 0, 1, 2 compared)

🎯 KEY LEARNINGS:

1. DELTA VERSIONING
   • Every write operation creates a new version
   • Versions are sequential (0, 1, 2, ...)
   • Retained for 30 days by default

2. TIME TRAVEL SYNTAX
   • versionAsOf: Query by version number
   • timestampAsOf: Query by date/time
   • Works with Python API and SQL

3. DATA RECOVERY
   • RESTORE TABLE: Rollback to previous version
   • Re-insert: Recover specific records
   • Audit trail: Track all changes

4. PRODUCTION USE CASES
   • Rollback bad deployments
   • Reproduce historical reports
   • Debug data quality issues
   • Comply with regulations

💡 BEST PRACTICES:

1. Configure retention period based on needs:
   ALTER TABLE SET TBLPROPERTIES ('delta.logRetentionDuration' = '90 days')

2. Document critical versions in MLflow:
   mlflow.log_param("data_version", 42)

3. Use DESCRIBE HISTORY for auditing:
   DESCRIBE HISTORY table_name

4. Run VACUUM periodically to clean up:
   VACUUM table_name RETAIN 168 HOURS

5. Test RESTORE in dev before using in production

⚠️ LIMITATIONS:

• Time travel only works within retention period
• VACUUM permanently deletes old versions
• Large tables may have slow time travel queries
• Schema changes may affect older versions

🔗 INTEGRATION WITH ML PIPELINE:

# Track data versions in experiments
data_version = spark.sql("DESCRIBE HISTORY table").first()["version"]
mlflow.log_param("data_version", data_version)

# Reproduce experiments exactly
data = spark.read.format("delta").option("versionAsOf", data_version).table("table")

🔗 NEXT STEPS:
• Day 12: Cost Optimization Strategies
• Day 13: Architecture Design
• Day 14: Production System
""")

print("\n✅ Day 11 Complete!")